In [42]:
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
import random
from torch.utils.data import TensorDataset, DataLoader

In [44]:
# Linear model (no bias)
class LinearModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, 1, bias=False)

    def forward(self, x):
        return self.linear(x)


In [46]:
# Local synthetic data
def generate_local_data(n_samples, true_beta, dim):
    X = torch.randn(n_samples, dim)
    y = X @ true_beta + 0.1 * torch.randn(n_samples, 1)
    return X, y

In [48]:
# Sample neighbors and normalize weights
def sample_neighbors(G: nx.Graph, k: int, sample_size: int):
    neighbors = list(G.neighbors(k)) + [k]  # include self
    if sample_size >= len(neighbors):
        sampled = neighbors
    else:
        sampled = random.sample(neighbors, sample_size)
    weights = {j: 1.0 / len(sampled) for j in sampled}
    return sampled, weights


In [50]:
# Main decentralized gradient descent with sampling
def decentralized_gd_with_sampling(data_loader, G: nx.Graph, dim=10, epochs=20, lr=0.1, sample_size=2):
    n_clients = G.number_of_nodes()
    models = [LinearModel(dim) for _ in range(n_clients)]
    params = [model.linear.weight.data.clone() for model in models]

    for epoch in range(epochs):
        for batch_iter in zip(*data_loader):
            grads = []

            # Step 1: Compute gradients
            for k, (X, y) in enumerate(batch_iter):
                models[k].linear.weight.data.copy_(params[k])
                models[k].zero_grad()
                pred = models[k](X).squeeze()
                loss = nn.MSELoss()(pred, y.squeeze())
                loss.backward()
                grads.append(models[k].linear.weight.grad.clone())

            # Step 2: Sample neighbors and compute local update
            new_params = []
            for k in range(n_clients):
                sampled_neighbors, weights = sample_neighbors(G, k, sample_size)
                avg_theta = sum(weights[j] * params[j] for j in sampled_neighbors)
                updated_theta = avg_theta - lr * grads[k]
                new_params.append(updated_theta)

            params = new_params  # Apply updates

        print(f"Epoch {epoch+1} complete.")

    return params

In [64]:
# Centralized training using full dataset
def centralized_gradient_descent(data_splits, dim=10, epochs=20, lr=0.1):
    # Merge all client datasets
    X_all = torch.cat([X for X, _ in data_splits], dim=0)
    y_all = torch.cat([y for _, y in data_splits], dim=0)
    loader = DataLoader(TensorDataset(X_all, y_all), batch_size=64, shuffle=True)

    model = LinearModel(dim)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            pred = model(X_batch).squeeze()
            loss = loss_fn(pred, y_batch.squeeze())
            loss.backward()
            optimizer.step()
        print(f"[Centralized] Epoch {epoch+1} Loss: {loss.item():.4f}")

    return model.linear.weight.data.clone()

In [52]:
num_clients = 5
dim = 10
true_beta = torch.ones(dim, 1)
data_n = 200

In [54]:
import matplotlib.pyplot as plt
G = nx.cycle_graph(num_clients)  # Ring topology
#nx.draw(G, with_labels=True)
#plt.title("Graph Topology: Ring")
#plt.show()


In [56]:
n_clients = num_clients
data = [generate_local_data(data_n, true_beta, dim = dim) for _ in range(n_clients)]
loaders = [DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True) for X, y in data]

In [60]:
decentral_theta = decentralized_gd_with_sampling(data_loader = loaders, G = G, sample_size=2)

[tensor([[1.0034, 1.0175, 0.9978, 0.9980, 0.9927, 0.9959, 0.9975, 1.0087, 1.0062,
          0.9924]]),
 tensor([[0.9981, 1.0058, 0.9965, 0.9979, 1.0074, 0.9924, 1.0018, 1.0053, 0.9986,
          1.0002]]),
 tensor([[1.0060, 0.9986, 1.0004, 0.9825, 1.0170, 0.9975, 1.0066, 0.9968, 1.0033,
          1.0070]]),
 tensor([[0.9980, 1.0081, 0.9990, 1.0021, 1.0108, 0.9958, 0.9934, 1.0039, 0.9973,
          1.0073]]),
 tensor([[1.0017, 0.9916, 1.0017, 0.9900, 0.9966, 0.9982, 1.0028, 1.0021, 0.9916,
          1.0114]])]

In [72]:
# Centralized training
central_theta = centralized_gradient_descent(data_splits = data , dim=dim, epochs=20, lr=0.1)

print("\nTrue β*:", true_beta.squeeze().numpy())
print("Estimated β (centralized):", central_theta.squeeze().numpy())

[Centralized] Epoch 1 Loss: 0.0332
[Centralized] Epoch 2 Loss: 0.0086
[Centralized] Epoch 3 Loss: 0.0064
[Centralized] Epoch 4 Loss: 0.0128
[Centralized] Epoch 5 Loss: 0.0074
[Centralized] Epoch 6 Loss: 0.0117
[Centralized] Epoch 7 Loss: 0.0089
[Centralized] Epoch 8 Loss: 0.0087
[Centralized] Epoch 9 Loss: 0.0157
[Centralized] Epoch 10 Loss: 0.0133
[Centralized] Epoch 11 Loss: 0.0143
[Centralized] Epoch 12 Loss: 0.0109
[Centralized] Epoch 13 Loss: 0.0134
[Centralized] Epoch 14 Loss: 0.0088
[Centralized] Epoch 15 Loss: 0.0153
[Centralized] Epoch 16 Loss: 0.0108
[Centralized] Epoch 17 Loss: 0.0073
[Centralized] Epoch 18 Loss: 0.0096
[Centralized] Epoch 19 Loss: 0.0080
[Centralized] Epoch 20 Loss: 0.0089

True β*: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Estimated β (centralized): [0.99887645 1.0035181  1.0015734  0.99892914 1.0094366  0.9978617
 0.9926891  1.0039839  0.9982103  0.99484444]
